# BarnacleAI: DALI Data Challenge 26F

**Research question:** Can two complementary ML approaches help count barnacles inside a green quadrat, and flag unreliable estimates for manual review?

This notebook is an executable narrative. It deliberately holds out an **entire photograph** from training. The blue mask annotations are **incomplete**, so all error values below compare to *annotation-derived proxies*, not verified population ground truth.

**Original student idea:** combine deep learning, linear regression, comparative error, human review and a 3D diagnostic.


In [ ]:
from pathlib import Path
import sys
cwd=Path.cwd().resolve()
project_root=cwd if (cwd/'barnacleai').is_dir() else cwd.parent
sys.path.insert(0,str(project_root))
import json
import numpy as np
import matplotlib.pyplot as plt
from barnacleai.data import ROOT, read_image, crop_roi, annotation_centers
from barnacleai.features import image_features, FEATURE_NAMES
print('Project root:',ROOT)

## 1. Problem and subtasks

The scientists need to record how many barnacles lie **inside the central (inner) green square**. Manually counting can be very slow. We split the project into:

1. Crop the correct square, excluding frame/ruler.
2. Derive approximate object centers from the supplied annotation masks.
3. Predict counts using a CNN and Ridge regression separately.
4. Evaluate on a held-out photograph; identify which tiles need human review.

We do **not** attempt to fully automate ground-truth collection.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,name in zip(axes,['img1','img2']):
    rgb=crop_roi(read_image(name),name)
    centers=annotation_centers(name)
    ax.imshow(rgb);ax.set_title(f'{name} — approximately {len(centers)} annotated centers');ax.axis('off')
plt.tight_layout()

## 2. COSC 3: turn pixels into interpretable features

In an 8×8 digit image, pixels may have grayscale intensity 0–16. Your lecture normalizes to `w = r/16`. In photographs we normalize 8-bit brightness to `w = r/255`.

- Density: $D = (1/HW)\sum_{a,b}w_{ab}$.
- Horizontal center: $\mu_x = \sum_b x_b w_{ab}/\sum_{a,b}w_{ab}$.
- Symmetry: $S = 1-\sum|w-F(w)|/(2\sum w)$.
- Balance: $B=(L-R)/(L+R)$.

These are **brightness** features, not a magical barnacle mask. Rock brightness can confound them, which is why we include edge/texture features and a separate CNN.


In [ ]:
uniform=np.full((8,8,3),128,dtype=np.uint8)
f=image_features(uniform)
for name,value in zip(FEATURE_NAMES,f):
    print(f'{name:24s}: {value:.4f}')
print('Check: density near 0.5, horizontal balance 0, symmetry 1')

## 3. Two models

**Model A — CNN:** A small convolutional network maps image pixels to a heatmap of possible barnacle *centers*. Peak detection turns heatmaps into individual counts. This model is trained from scratch as a prototype (no downloaded pretrained weights).

**Model B — Ridge regression:** A standardized numerical feature vector predicts how many barnacles should appear per 128×128 tile. Ridge controls unstable coefficients that can occur when features correlate.

**Train/test rule:** train both models on one annotated image and evaluate on the other. Avoid random mixing of tiles from the two photos, because that produces data leakage.

To rerun training, execute:

```bash
python -m barnacleai.train --epochs 3 --max-train-tiles 75
```


In [ ]:
metrics_path=ROOT/'results'/'metrics.json'
if metrics_path.exists():
    metrics=json.loads(metrics_path.read_text())
    print(json.dumps({k:v for k,v in metrics.items() if k not in ('feature_names','loss_by_epoch')},indent=2))
else:
    print('Models not trained. Run python -m barnacleai.train first.')

## 4. Evaluation and failure analysis

- **Mean absolute error** (MAE): average of $|\hat{y}_i-y_i|$ over nonoverlapping 128×128 tiles, using proxy annotation counts as $y_i$.
- **Two-model disagreement**: $|\hat{y}_{CNN}-\hat{y}_{Ridge}|$ per tile. Tiles with large discrepancies should be manually reviewed.
- **Not a confidence interval**: the model's predicted probabilities and its disagreement do **not** establish statistically calibrated coverage. Calibrated 95% count intervals require many independent, representative labeled photographs and a specific calibration procedure.

Look at the saved test overlay and inspect false positives / missed detections. The blue mask annotations may themselves miss real barnacles. Are we measuring biology, or annotation quality?


In [ ]:
from PIL import Image
path=ROOT/'results'/'heldout_overlay.png'
if path.exists():
    plt.figure(figsize=(9,9));plt.imshow(Image.open(path));plt.axis('off')
    plt.title('Held-out CNN detections (green); model disagreement review boxes (orange)');plt.show()
else:
    print('First train the models to produce this plot.')

## 5. 3D visualization and human feedback

X and Y are tile positions in the photograph; Z is the magnitude of disagreement between the CNN and Ridge counts for that tile. This is a *diagnostic*, not a third physical dimension of the barnacles and not a calibrated uncertainty estimate. A scientist can inspect flagged patches and provide the corrected count through the `/feedback` endpoint.


In [ ]:
from IPython.display import display, HTML
from pathlib import Path
chart=ROOT/'results'/'model_disagreement_3d.html'
if chart.exists():
    print('Interactive local HTML saved at:',chart)
else:
    print('Run training to create the interactive 3D HTML diagnostic.')

## 6. Conclusion, next experiments, and learning

This is a deliberately limited pilot. Results may be weak because of different barnacle density, contrast, species/appearance, and only two labeled photographs. We prefer honest bad results to inflated train/test performance.

**What I learned:** Feature engineering can reduce an image to meaningful scalar measures, but those measures lose spatial detail. CNNs can extract visual structure but need diverse training examples. Splitting by *independent photographs* rather than randomly by tile matters tremendously for realistic evaluation.

**Next experiments:** manually correct small independent validation crops, collect more labeled photos, compare pretrained object segmentation models, reduce false positives, and quantify how much manual labor the prototype saves.

**AI assistance disclosure:** Implementation assistance was used for code scaffolding, model setup and debugging. The researcher must inspect, run and explain the pipeline and conclusions.


## 7. Designing a statistically defensible counting interval
Model heatmap scores are not calibrated confidence intervals. An alternative is to **randomly sample** tiles for independent human verification, then infer the photographed square’s total.

Divide the region into 64 nearly equal-size tiles, sample 12 tiles **uniformly without replacement**, and manually count each tile. Let $y_i$ be verified counts and $m_i$ be CNN counts.

Direct estimator: $\hat T = N\bar y$; model-assisted difference estimator: $\hat T = \sum_i m_i + N\overline{(y-m)}$.

A t-approximate 95% interval adds and subtracts $t_{0.975,n-1}N\sqrt{(1-n/N)s^2/n}$, with $s^2$ the sample variance of counts (or sampled residuals for CNN assistance). Assumes random sampling and accurate human counts.

To check the code, substitute incomplete *annotation counts* in the sampled tiles, but that produces **only an interval about the incomplete annotations**, NOT about real barnacle numbers.


In [ ]:
from barnacleai.uncertainty import partition_image,random_tile_indices,counts_per_box,finite_population_interval
from barnacleai.data import annotation_centers
img=crop_roi(read_image("img2"),"img2")
grid=partition_image(*img.shape[:2])
sample_ids=random_tile_indices(len(grid),12)
annotation_counts=counts_per_box(annotation_centers("img2"),grid)
demo=finite_population_interval(annotation_counts[sample_ids],sample_ids,len(grid))
print("Demonstration about incomplete blue annotations ONLY:",demo)
print("True blue-annotation-proxy total (not biological ground truth):",sum(annotation_counts))


## 8. What we cannot claim yet

The real biological 95% confidence interval remains **pending** until twelve randomly selected tiles have independently verified manual counts. No amount of optimistic model confidence replaces those data. Open `results/random_manual_review_img2.png` to review the tiles, and follow `RESULTS.md`.
